#### Restart & Run All | Send orders.csv to obsidian

In [1]:
import calendar
import pandas as pd
import sidetable
import panel as pn
import seaborn as sns
from datetime import date, timedelta
from sqlalchemy import create_engine
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=False)
pn.extension('tabulator')

engine = create_engine("sqlite:///c:\\ruby\\port_lite\\db\\development.sqlite3")
conlite = engine.connect()

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"
osd_path = "\\Users\\User\\OneDrive\\Documents\\obsidian-git-sync\\Data\\"

today = date.today()
today

<IPython.core.display.Javascript object>

datetime.date(2023, 1, 27)

### Begin of Tables in the process

In [2]:
cols = 'trade name qty price active reason market xdate'.split()
colt = 'trans name qty target active spd current change percent reason market xdate'.split()

format_dict = {
    'qty':'{:,}',
    'price':'{:.2f}','target':'{:.2f}','current':'{:.2f}','change':'{:.2f}','diff':'{:.2f}',
    'amount':'{:,.2f}','sell_amt':'{:,.2f}'
}

pd.read_sql_query('SELECT * FROM orders ORDER BY id DESC LIMIT 1', conlite).style.format(format_dict)

,id,trade,name,qty,price,active,reason,market,xdate
0,21,B,JMT,900,53.00,1,RD10pct,SET50,2022-02-02


### Print to verify before upload file

In [3]:
sql = '''
SELECT trade, name, qty, price, qty * price AS amount, reason, market, active, xdate
FROM orders
ORDER BY trade, name'''
orders = pd.read_sql(sql, conlite)

df_tab = pn.widgets.Tabulator(orders, layout='fit_data', width=710)
df_tab

Tabulator(layout='fit_data', value=   trade     n..., width=710)

In [4]:
orders.stb.freq(["market"])

,market,count,percent,cumulative_count,cumulative_percent
0,SET,8,38.095238,8,38.095238
1,SET100,7,33.333333,15,71.428571
2,SET50,6,28.571429,21,100.000000


In [5]:
file_name = 'orders.csv'
data_file = data_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name
one_file = one_path + file_name
osd_file = osd_path + file_name

orders[cols].to_csv(output_file, header=True, index=False)
orders[cols].to_csv(data_file, header=True, index=False)
orders[cols].to_csv(box_file, header=True, index=False)
orders[cols].to_csv(one_file, header=True, index=False)
orders[cols].to_csv(osd_file, header=True, index=False)

### End of transactional process

### Process to set target price

#### 1) Set50 records

In [6]:
pd.read_sql_query('SELECT * FROM orders WHERE market = "SET50" ORDER BY trade, name', conlite)

,id,trade,name,qty,price,active,reason,market,xdate
0,12,B,IVL,2400,36.0,0,RD15pct,SET50,2023-05-06
1,14,B,JMART,1200,33.5,0,RD15pct,SET50,2023-04-19
2,21,B,JMT,900,53.0,1,RD10pct,SET50,2022-02-02
3,7,B,TTB,90000,1.4,2,52WL,SET50,2022-12-22
4,9,S,BANPU,12000,13.0,2,DTD,SET50,2023-04-08
5,16,S,SCC,300,360.0,0,5pct,SET50,2023-04-07


In [7]:
name = 'JMART'
limit = 20 # 1 month of data
sql = """
SELECT P.*
FROM price P
WHERE P.name = '%s'
ORDER BY date DESC
LIMIT %s"""
sql = sql % (name, limit)
df = pd.read_sql(sql, const)
df.describe().round(2)

,price,maxp,minp,qty,opnp
count,20.00,20.00,20.00,20.00,20.00
mean,40.45,41.26,40.10,9252301.95,40.83
std,1.76,1.41,1.66,7717467.91,1.28
min,35.50,38.00,35.25,3095538.00,37.75
25%,39.75,40.38,39.38,4919119.50,40.00
50%,40.88,41.50,40.62,6358168.00,41.00
75%,41.56,42.31,41.12,10501846.25,41.75
max,42.50,43.50,42.00,35425033.00,42.50


#### 2) Set100 records

In [8]:
pd.read_sql_query('SELECT * FROM orders WHERE market = "SET100" ORDER BY trade, name', conlite)

,id,trade,name,qty,price,active,reason,market,xdate
0,1,B,CK,4500,22.50,1,5pct,SET100,2022-12-22
1,2,B,CKP,20000,4.50,1,3L,SET100,2022-12-22
2,6,B,MEGA,2000,48.00,0,5pct,SET100,2023-03-08
3,10,S,BCH,3000,22.50,0,5pct,SET100,2023-05-10
4,15,S,ORI,15000,12.50,1,CP1S,SET100,2023-05-09
5,17,S,SINGER,3600,29.25,1,DTD,SET100,2023-04-29
6,18,S,STA,2500,23.00,2,DTD,SET100,2023-04-19


In [9]:
name = 'SINGER'
limit = 60 # 3 month of data
sql = """
SELECT P.*
FROM price P
WHERE P.name = '%s'
ORDER BY date DESC
LIMIT %s"""
sql = sql % (name, limit)
df = pd.read_sql(sql, const)
df.describe().round(2)

,price,maxp,minp,qty,opnp
count,60.00,60.00,60.00,60.00,60.00
mean,30.83,31.54,30.35,6087109.87,31.06
std,2.79,2.90,2.80,3712277.10,2.84
min,26.75,27.25,26.25,1984596.00,27.00
25%,28.50,29.25,28.00,3324422.50,28.50
50%,30.25,31.38,30.00,5274103.00,30.62
75%,32.25,32.75,32.00,7381591.75,32.50
max,37.00,37.50,36.25,20019390.00,37.00


#### 3) Set records

In [10]:
# Why SIS does not print out from this SQL?
pd.read_sql_query('SELECT * FROM orders WHERE market = "SET" ORDER BY trade, name', conlite)

,id,trade,name,qty,price,active,reason,market,xdate
0,3,B,CPNREIT,10000,18.7,1,ROUND,SET,2023-03-01
1,4,B,GFPT,3600,12.5,0,DTD,SET,2023-03-07
2,5,B,MC,10000,11.0,1,HD,SET,2023-02-24
3,8,B,WHAIR,10000,7.7,1,RD15pct,SET,2023-03-04
4,11,S,DIF,10000,14.0,1,CUT,SET,2023-02-10
5,13,S,JASIF,10000,8.5,1,CP1S,SET,2023-03-03
6,19,S,SYNEX,3000,17.5,0,5pct,SET,2023-03-10
7,20,S,TMT,9000,8.7,1,C9.0,SET,2023-04-18


In [11]:
name = 'TMT'
limit = 120 # 6 month of data
sql = """
SELECT P.*
FROM price P
WHERE P.name = '%s'
ORDER BY date DESC
LIMIT %s"""
sql = sql % (name, limit)
df = pd.read_sql(sql, const)
df.describe().round(2)

,price,maxp,minp,qty,opnp
count,120.00,120.00,120.00,120.00,120.00
mean,7.76,7.81,7.69,354131.64,7.76
std,0.41,0.40,0.40,314295.97,0.40
min,6.95,7.05,6.95,54095.00,7.00
25%,7.55,7.60,7.50,159673.00,7.59
50%,7.85,7.95,7.80,299074.50,7.85
75%,8.00,8.05,7.95,422134.00,8.00
max,8.45,8.55,8.35,2565557.00,8.50


In [12]:
pd.read_sql_query('SELECT * FROM orders ORDER BY trade, name', conlite)

,id,trade,name,qty,price,active,reason,market,xdate
0,1,B,CK,4500,22.50,1,5pct,SET100,2022-12-22
1,2,B,CKP,20000,4.50,1,3L,SET100,2022-12-22
2,3,B,CPNREIT,10000,18.70,1,ROUND,SET,2023-03-01
3,4,B,GFPT,3600,12.50,0,DTD,SET,2023-03-07
4,12,B,IVL,2400,36.00,0,RD15pct,SET50,2023-05-06
5,14,B,JMART,1200,33.50,0,RD15pct,SET50,2023-04-19
6,21,B,JMT,900,53.00,1,RD10pct,SET50,2022-02-02
7,5,B,MC,10000,11.00,1,HD,SET,2023-02-24
8,6,B,MEGA,2000,48.00,0,5pct,SET100,2023-03-08
9,7,B,TTB,90000,1.40,2,52WL,SET50,2022-12-22
